# Laboratorijske vaje 12

## 1. Načrtovanje optimalnih KEO filtrov z linearno faze po Čebiševem kriteriju 
S pomočjo Parks-McClellanove metode načrtuj naslednja filtra:
- $N=41$, enakomerne uteži (1), z naslednjimi mejami prepustnega in zapornega pasu.
- **Filter 1**: Meja med prepustnim in zapornim pasom je točka z relativno frekvenco 0.25. 
- **Filter 2**: Prepustni pas [0, 0.24], zaporni pas [0.26, 0.5].

V Pythonu lahko optimalen filter po Parks-McClellanovi metodi načrtamo s funkcijo ```sgn.remez``` ([dokumentacija](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.remez.html)).

Naloge: 
- Analizirajte amplitudne odzive filtrov (funkcija ```sgn.freqz```). 
- Določite frekvence mejnih točk za primera: $F_s=22050 Hz$, $F_s=8000Hz$.
- Razmislite o smiselnosti uvedbe prehodnega pasu.
- Opazujte kaj se dogaja z amplitudnim odzivom v odvisnosti od števila koeficientov filtra $N$. 


In [ ]:
%matplotlib ipympl
import numpy as np
import scipy.signal as sgn
import matplotlib.pyplot as plt


# Praktični primeri

## P1. Načrtovanje komplementarnih filtrov za zvočni signal
V datoteki *yesterday_sinus.wav* se nahaja posnetek melodije, zgenerirane s pomočjo enostavnega sinusnega sintetizatorja – torej vsaka nota je predstavljena z natanko enim, njej ustreznim čistim tonom. Melodija je sestavljena iz dveh delov: 
- *Solo*, ki sega od note G4 navzgor (področje od približno 380Hz navzgor).
- *Spremljava*, ki sega do note E4 (področje do približno 290Hz).

Pri 1. nalogi ste se naučili načrtovati optimalne KEO filtre z linearno fazo. Načrtujte 2 filtra tako, da bosta komplementarna (vsak "prepusti" tisti del spektra, ki ga drugi "zaduši") z naslednjimi podatki:
- $N=101,501,801$, utež v zapornem pasu 10, mejni točki sta pri 290 Hz in 380 Hz.
- Preverite dobljene rezultate s pomočjo amplitudnega odziva in s poslušanjem rezultatov delovanja filtra. 
- Primerjajte lastnosti rešitev pri različnih dolžinah filtrov (101, 501, 801).

In [ ]:
%matplotlib ipympl
import numpy as np
import scipy.signal as sgn
import matplotlib.pyplot as plt
from scipy.io import wavfile
import sounddevice as sd


# P2. Načrtovanje filtrov za spektralni izenačevalnik
Se še spomnite opisa simulacije spektralnega izenačevalnika?

Pri 1. nalogi ste se naučili načrtovati optimalne KEO filtre z linearno fazo. Enak postopek uporabite za načrtovanje filtrov spektralnega izenačevalnika:
- Uporabite poljubno število filtrov, ki bo najmanj 3 in različno od 5.
- Preverite dobljene rezultate s pomočjo amplitudnega odziva in praktične uporabe filtrov na zvočni datoteki. 
- Vsi filtri naj imajo enako dolge odzive na enotin impulz ($N=101$). Nasvet: priporočamo enako širino prehodnih pasov pri načrtovanju filtrov (levo in desno od prepustnega pasu).

## Tipična razdelitev za 5 filtrov

Za 5-pasovni spektralni izenačevalnik je smiselno izbrati filtre tako, da približno pokrijejo glavna področja človeškega sluha in tipične lastnosti glasbe/govora.
Najpogostejši pristop je **logaritmična razporeditev frekvenc**, ker tudi človeško uho zaznava frekvence približno logaritmično.

|Pas | Spodnja frekvenca | Zgornja frekvenca |
|---|---|---|
| Nizki | 20 Hz | 250 Hz | 
| Srednji | 250 Hz | 1000 Hz |
| Visoki srednji | 1000 Hz | 8000 Hz |
| Visoki | 8000 Hz | 20000 Hz|

In [ ]:
%matplotlib ipympl
import numpy as np
import scipy.signal as sgn
import matplotlib.pyplot as plt
from scipy.io import wavfile


## Načtovanje NEO (IIR) filtrov s knjižnico SciPy

V knjižnici SciPy obstaja več vrst NEO filtrov, ki se razlikujejo po svojih lastnostih. 

| Filter       | Valovitost            | Strmina prehoda | Fazni odziv | Tipična uporaba            |
| ------------ | ----------------- | --------------- | ----------- | -------------------------- |
| Butterworth  | ne                | srednja         | dober       | splošna uporaba            |
| Chebyshev I  | v prepustnem pasu | velika          | srednji     | ozki filtri                |
| Chebyshev II | v dušilnem pasu   | velika          | srednji     | odstranjevanje šuma        |
| Elliptic     | v obeh pasovih    | največja        | slabši      | optimizirani DSP           |
| Bessel       | ne                | majhna          | najboljši   | časovno občutljivi signali |

Primer uporabe:
```python
butter(N, Wn, btype='low', analog=False, output='ba', fs=None)
# N - red filtra
# Wn - mejna frekvenca/array frekvenc
# btype {‘lowpass’, ‘highpass’, ‘bandpass’, ‘bandstop’} - tip filtra
# output - {‘ba’, ‘zpk’, ‘sos’} - izhodna oblika
# fs - vzorčevalna frekvenca
```

Koda spodaj prikaže razlike med posameznimi tipi filtrov na primeru nizkoprepustnega filtra.

In [ ]:
%matplotlib ipympl
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sgn

# Parametri
fs = 1000            # vzorčevalna frekvenca [Hz]
fc = 100              # mejna frekvenca LP filtra [Hz]
order = 4             # red filtra

# Normalizirana frekvenca
wn = fc / (fs / 2)

# Definicija filtrov
filters = { "Butterworth": sgn.butter(order, wn, btype='low'),
            "Chebyshev I": sgn.cheby1(order, 1, wn, btype='low'),
            "Chebyshev II": sgn.cheby2(order, 40, wn, btype='low'),
            "Elliptic": sgn.ellip(order, 1, 40, wn, btype='low'),
            "Bessel": sgn.bessel(order, wn, btype='low', norm='phase')
}

plt.figure()
for name, (b, a) in filters.items():

    # Frekvenčni odziv
    f, h = sgn.freqz(b, a, worN=1024, fs=fs)
    magnitude = np.abs(h)
    phase = np.unwrap(np.angle(h))

    plt.subplot(2, 1, 1)
    plt.plot(f, magnitude, label=name)
    
    plt.subplot(2, 1, 2)
    plt.plot(f, phase, label=name)

# Amplitudni odziv
plt.subplot(2, 1, 1)
plt.title("Amplitudni odziv IIR LP filtrov")
plt.ylabel("Amplituda [dB]")
plt.grid(True)
plt.legend()

# Fazni odziv
plt.subplot(2, 1, 2)
plt.title("Fazni odziv IIR LP filtrov")
plt.ylabel("Faza [rad]")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# Orodje pyFDA (Filter Design & Analysis Tool)

https://github.com/chipmuenk/pyfda

![pyFDA](pyFDA.png)